In [7]:
import pandas as pd
import re
import csv
import string
import math

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from pathlib import Path

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

import spacy
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 1500000 


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ftzavellos/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ftzavellos/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/ftzavellos/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
foe_df_path = ".." / Path.cwd().parent / "dataset" / "foe_echr.csv"
df_dataset = pd.read_csv(foe_df_path, index_col=0, encoding = 'utf-8')


In [24]:
print('Number of nan values in the facts column of the foe dataset is' , df_dataset[['facts']].isna().sum())
df_dataset = df_dataset.dropna(subset=['facts'])
df_dataset = df_dataset.reset_index(drop=True)


Number of nan values in the facts column of the foe dataset is facts    40
dtype: int64


In [25]:
def clean_text(text):
    # Remove markdown elements and special characters
    text = re.sub(r'#', '', text)  # Remove '###'
    text = re.sub(r'[-]', '', text)  # Remove '-'
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = re.sub(r'\\[a-zA-Z0-9]+', '', text)  # Remove any escaped sequences like \xa0
    return text

In [26]:
df_dataset['facts'] = df_dataset['facts'].apply(clean_text)
df_dataset['full_text'] = df_dataset['full_text'].apply(clean_text)

In [28]:
ne_path = ".." / Path.cwd().parent / "dataset" / "named_entities_foe.csv"

In [27]:
facts = df_dataset['facts']
facts_list = facts.tolist()
facts_list = [str(fact) for fact in facts_list]

# Extract named entities from facts_list
entities = [ent.text for doc in map(nlp, facts_list) for ent in doc.ents]

# Get the unique values from the entities list
entities_list = list(dict.fromkeys(entities))

# Save the list of entities
with open(ne_path, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    for ent in entities_list:
        writer.writerow([ent])


In [29]:
entities_facts  = pd.read_csv(ne_path, header = None, names = ['facts'], encoding = 'utf-8')
entities_list_facts = entities_facts['facts'].tolist()
entities_list_facts = [str(element) for element in entities_list_facts]
entities_list_facts = [x for x in entities_list_facts if x.lower() != "nan"]
print(sum(1 for x in entities_list_facts if x.lower() == "nan"))

0


In [30]:
def remove_named_entities(text, entities_list):
    '''
    
    '''
    for entity in entities_list:
        # Use word boundaries to ensure whole phrases are matched
        entity_regex = re.compile(r'\b' + re.escape(entity) + r'\b', re.IGNORECASE)
        text = entity_regex.sub('', text)
    return text.strip()

In [31]:
df_dataset['facts_ne_removed'] = df_dataset['facts'].apply(lambda x: remove_named_entities(x, entities_list_facts))

In [36]:
all_stopwords = stopwords.words('english')
all_stopwords.extend([
'also',
'may', 
'could', 
'would', 
'must', 
'applicant', 
'applicants'
'court',
'article',
'case',
'convetion',
'see',
'right',
'government',
'paragraph',
'law',
'state',
'detention',
'authority',
'application',
'one'])


In [37]:
def remove_stopwords(text, stopwords_list):
    for entity in stopwords_list:
        # Use word boundaries to ensure whole phrases are matched
        stopword_regex = re.compile(r'\b' + re.escape(entity) + r'\b', re.IGNORECASE)
        text = stopword_regex.sub('', text)
    return text.strip()


In [38]:
df_dataset['cleaned_facts'] = df_dataset['facts_ne_removed'].apply(lambda x: remove_stopwords(x, all_stopwords))

In [35]:
dataset_path = ".." / Path.cwd().parent / "dataset" / "preprocessed_dataset.csv"
df_dataset.to_csv(dataset_path)

In [39]:
df_dataset

,itemid,docname,article,appno,judgementdate,law,facts,the_conclusion,full_text,respondent,judgementdate.1,facts_ne_removed,cleaned_facts
0,001-209033,CASE OF HANDZHIYSKI v. BULGARIA,"['35', '41', '10']",10783/14,06/04/2021 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLE ...,THE FACTS 2. The applicant was born in 1971 a...,"### FOR THESE REASONS, THE COURT\n\n- Declares...",FOURTH SECTIONCASE OF HANDZHIYSKI v. BULGARIA(...,BGR,06/04/2021 00:00:00,THE . The was born in and lives in . He was...,". born lives . represented , lawye..."
1,001-57513,CASE OF KOSIEK v. GERMANY,['10'],9704/82,28/08/1986 00:00:00,### AS TO THE LAW\n\n- I. THE GOVERNMENT’S P...,"AS TO THE FACTS 11. Mr. Rolf Kosiek, who is a...","### FOR THESE REASONS, THE COURT\n\n- Holds by...",COURT (PLENARY) CASE OF KOSIEK v. GERMANY (App...,DEU,28/08/1986 00:00:00,". . , is a born in , lives in . After study...",". . , born , lives . studying physics ..."
2,001-155196,CASE OF MEHDIYEV v. AZERBAIJAN,"['41', '3', '5', '35', '10']",59075/09,18/06/2015 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,THE FACTS I. THE CIRCUMSTANCES OF THE CASE 5....,"### FOR THESE REASONS, THE COURT\n\n- 1. Decl...",FIRST SECTION CASE OF MEHDIYEV v. AZERBAIJAN (...,AZE,18/06/2015 00:00:00,THE CIRCUMSTANCES OF THE CASE . The was bor...,CIRCUMSTANCES . born lived eve...
3,001-84268,CASE OF FEVZİ SAYGILI v. TURKEY,"['14', '10', '13']",74243/01,08/01/2008 00:00:00,### THE LAW\n\n- I. THE GOVERNMENT'S PRELIMIN...,THE FACTS I. THE CIRCUMSTANCES OF THE CASE 5....,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",SECOND SECTION CASE OF FEVZİ SAYGILI v. TURKEY...,TUR,08/01/2008 00:00:00,THE CIRCUMSTANCES OF THE CASE . The was bor...,CIRCUMSTANCES . born lives . . o...
4,001-107591,CASE OF KILIÇ AND EREN v. TURKEY,['10'],43807/07,29/11/2011 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,THE FACTS I. THE CIRCUMSTANCES OF THE CASE 5....,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",SECOND SECTION CASE OF KILIÇ AND EREN v. TURKE...,TUR,29/11/2011 00:00:00,THE CIRCUMSTANCES OF THE CASE . The applican...,CIRCUMSTANCES . applicants born respe...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,001-218117,CASE OF DROUSIOTIS v. CYPRUS,"['8', '41', '10']",42315/15,05/07/2022 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLE ...,THE FACTS 2. The applicant was born in 1959 a...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",THIRD SECTIONCASE OF DROUSIOTIS v. CYPRUS(Appl...,CYP,05/07/2022 00:00:00,THE . The was born in and lives in . He was...,. born lives . represented Ms . C...
925,001-61716,CASE OF AMIHALACHIOAIE v. MOLDOVA,"['41', '10']",60115/00,20/04/2004 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,THE FACTS I. THE CIRCUMSTANCES OF THE CASE 8....,"### FOR THESE REASONS, THE COURT\n\n- 1. Hold...",SECOND SECTION CASE OF AMIHALACHIOAIE v. MOLDO...,MDA,20/04/2004 00:00:00,THE CIRCUMSTANCES OF THE CASE . The is a ...,CIRCUMSTANCES . born lives ()....
926,001-201087,CASE OF RELIGIOUS COMMUNITY OF JEHOVAH'S WITNE...,['10'],52884/09,20/02/2020 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLEs...,THE FACTS THE CIRCUMSTANCES OF THE CASE 6. Th...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",FIFTH SECTIONCASE OF RELIGIOUS COMMUNITY OF JE...,AZE,20/02/2020 00:00:00,THE THE CIRCUMSTANCES OF THE CASE . The is ...,"CIRCUMSTANCES . Religious , register..."
927,001-58032,"CASE OF X, Y AND Z v. THE UNITED KINGDOM","['8', '10', '14']",21830/93,22/04/1997 00:00:00,### AS TO THE LAW\n\n- I. ALLEGED VIOLATION ...,AS TO THE FACTS I. Circumstances of the case ...,"### FOR THESE REASONS, THE COURT\n\n- 1. Hol...","COURT (GRAND CHAMBER) CASE OF X, Y AND Z v. TH...",GBR,22/04/1997 00:00:00,I. Circumstances of the case . The applicants ...,". Circumstances . applicants , resident ..."
